In [ ]:
!pip install -q kagglehub
import os, shutil, json, kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models

path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
print("Path:", path)

for out in ["data_name", "data_health"]:
    shutil.rmtree(out, ignore_errors=True)
LIMIT, n = 100, 0
for root, dirs, files in os.walk(path):
    folder = os.path.basename(root).lower()
    if not (folder.startswith("fresh") or folder.startswith("rotten")):
        continue
    health = "rotten" if folder.startswith("rotten") else "fresh"
    fruit = folder.replace("fresh", "").replace("rotten", "").strip("_ -")
    if fruit.endswith("s"):
        fruit = fruit[:-1]
    imgs = [f for f in sorted(files) if f.lower().endswith((".jpg", ".jpeg", ".png"))][:LIMIT]
    for f in imgs:
        n += 1
        for d in (os.path.join("data_name", fruit), os.path.join("data_health", health)):
            os.makedirs(d, exist_ok=True)
            shutil.copy(os.path.join(root, f), os.path.join(d, f"{n}_{f}"))
print("Total images:", n)
print(os.listdir("data_name"), os.listdir("data_health"))

def train(data_dir, prefix, epochs=2):
    size = (224, 224)
    tr = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training",
        seed=42, image_size=size, batch_size=16)
    va = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation",
        seed=42, image_size=size, batch_size=16)
    names = tr.class_names
    json.dump(names, open(f"{prefix}_classes.json", "w"))
    base = tf.keras.applications.MobileNetV2(
        input_shape=size + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    inp = layers.Input(shape=size + (3,))
    x = layers.RandomFlip("horizontal")(inp)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(names), activation="softmax")(x)
    model = models.Model(inp, out)
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(tr, validation_data=va, epochs=epochs)
    model.save(f"{prefix}.keras")

train("data_name", "fruit_name")
train("data_health", "fruit_health")
print("TRAINING DONE")

from google.colab import drive
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/fruit_project", exist_ok=True)
for fn in ["fruit_name.keras", "fruit_health.keras",
           "fruit_name_classes.json", "fruit_health_classes.json"]:
    shutil.copy(fn, "/content/drive/MyDrive/fruit_project/" + fn)
print("SAVED TO DRIVE")

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.
Path: /kaggle/input/fruits-fresh-and-rotten-for-classification
Total images: 2400
['apple', 'orange', 'banana'] ['rotten', 'fresh']
Found 2400 files belonging to 3 classes.
Using 1920 files for training.
Found 2400 files belonging to 3 classes.
Using 480 files for validation.
Epoch 1/2
120/120 ━━━━━━━━━━━━━━━━━━━━ 20s 97ms/step - accuracy: 0.9177 - loss: 0.2213 - val_accuracy: 0.9875 - val_loss: 0.0578
Epoch 2/2
120/120 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.9896 - loss: 0.0477 - val_accuracy: 0.9937 - val_loss: 0.0312
Found 2400 files belonging to 2 classes.
Using 1920 files for training.
Found 2400 files belonging to 2 classes.
Using 480 files for validation.
Epoch 1/2
120/120 ━━━━━━━━━━━━━━━━━━━━ 20s 104ms/step - accuracy: 0.8849 - loss: 0.2738 - val_accuracy: 0.9833 - val_loss: 0.0862
Epoch 2/2
120/120 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.9719 - loss: 0.0961 - val_ac

In [ ]:
!pip install -q gradio

import json
import numpy as np
import tensorflow as tf
from PIL import Image
import gradio as gr
from google.colab import drive

drive.mount('/content/drive')
D = "/content/drive/MyDrive/fruit_project/"
name_model = tf.keras.models.load_model(D + "fruit_name.keras")
health_model = tf.keras.models.load_model(D + "fruit_health.keras")
name_cls = json.load(open(D + "fruit_name_classes.json"))
health_cls = json.load(open(D + "fruit_health_classes.json"))
print("Models loaded:", name_cls, health_cls)

NAME_MIN = 70
HEALTH_MIN = 70


def show(text, color):
    return "<h1 style='color:%s;text-align:center'>%s</h1>" % (color, text)


def analyse(frame):
    if frame is None:
        return show("Show a fruit to the camera", "gray")
    img = Image.fromarray(frame).convert("RGB").resize((224, 224))
    x = np.expand_dims(np.array(img, dtype="float32"), 0)
    pn = name_model(x, training=False).numpy()[0]
    ph = health_model(x, training=False).numpy()[0]

    fruit = name_cls[int(np.argmax(pn))]
    c1 = float(pn.max()) * 100
    health = health_cls[int(np.argmax(ph))]
    c2 = float(ph.max()) * 100

    if c1 < NAME_MIN:
        return show("No supported fruit found", "gray")
    if c2 < HEALTH_MIN:
        return show("%s - not sure, move closer" % fruit.title(), "orange")
    color = "green" if health == "fresh" else "red"
    text = "%s - %s (%d%%)" % (fruit.title(), health.upper(), round(c2))
    return show(text, color)


with gr.Blocks() as demo:
    gr.Markdown("# Fruit Health Detection - Live Camera")
    cam = gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Camera")
    result = gr.HTML()
    cam.stream(analyse, [cam], [result])

demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models loaded: ['apple', 'banana', 'orange'] ['fresh', 'rotten']
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa1a2ef0e8d25657fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!mkdir -p /content/drive/MyDrive/fruit_project

name_model.save('/content/drive/MyDrive/fruit_project/phase1_name_model.keras')
health_model.save('/content/drive/MyDrive/fruit_project/phase1_health_model.keras')

In [ ]:
from tensorflow.keras.models import load_model

name_model = load_model('/content/drive/MyDrive/fruit_project/phase1_name_model.keras')
health_model = load_model('/content/drive/MyDrive/fruit_project/phase1_health_model.keras')